# Project 01 (basic) — Multilingual subword tokenization with SentencePiece

**Module 10 — Multilingual NLP** · Format: **Jupyter notebook** (`tokenization.ipynb`)

Before a multilingual model learns even a single weight, text has to be split into tokens
— and *how* you do that decides the OOV rate, the sequence length and whether languages can
**share** representations. This project makes tokenization tangible: you train a **shared
subword vocabulary** on German **and** English and measure empirically what script section 2
claims.

Core questions you answer:

- What does a **BPE** split look like, and how does it handle unknown words?
- What is **fertility** (avg. subword tokens per word), and how does it differ between
  languages?
- How strongly do German and English **share** tokens in a shared vocabulary?
- What happens when the vocabulary is **English-dominated** (the *fairness* aspect of large
  LLMs)?

> **Plenty of instruction:** download, parsing and the SentencePiece calls are given. Your
> tasks hit the analysis cores: compute fertility, measure vocabulary sharing and show the
> vocabulary bias experimentally.


## Setup

Requires `sentencepiece` (in the repo `requirements.txt`). The first cell downloads the
**Tatoeba** German–English sentence-pair dataset (~12 MB) into `datasets/` and caches it.

```bash
source ../../../../.venv/bin/activate
jupyter lab      # or open tokenization.ipynb in VS Code, kernel = repo .venv
```

Everything runs in **under a minute** (SentencePiece is in C++ and very fast).


## Part A — Loading the data & preparing the corpus *(given)*

**Tatoeba** is an open collection of translation sentence pairs. We use the German–English
pairs (format per line: `english \t german \t attribution`). For fast training we write the
first ten thousands of pairs into a corpus file.


In [1]:
# ---- Load Tatoeba DE-EN (real translation pairs) --------------------------
import os, io, zipfile, urllib.request

DATA_DIR = "datasets"
os.makedirs(DATA_DIR, exist_ok=True)
RAW = os.path.join(DATA_DIR, "deu.txt")
URL = "https://www.manythings.org/anki/deu-eng.zip"

if not os.path.exists(RAW):
    print("Downloading Tatoeba DE-EN ...")
    # The site requires browser-like headers (otherwise HTTP 406).
    hdr = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
           "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
           "Accept-Language": "en-US,en;q=0.9",
           "Referer": "https://www.manythings.org/anki/"}
    req = urllib.request.Request(URL, headers=hdr)
    raw = urllib.request.urlopen(req, timeout=60).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with z.open("deu.txt") as src, open(RAW, "wb") as dst:
            dst.write(src.read())
    print("Done.")
else:
    print("Dataset already present.")

lines = open(RAW, encoding="utf-8").read().strip().split("\n")
pairs = [ln.split("\t")[:2] for ln in lines]        # [english, german]
print(f"{len(pairs):,} sentence pairs.")
for en, de in pairs[500:503]:
    print(f"  EN: {en}\n  DE: {de}\n")

Dataset already present.


331,266 sentence pairs.
  EN: I'm bald.
  DE: Ich habe eine Glatze.

  EN: I'm busy.
  DE: Ich bin beschäftigt.

  EN: I'm busy.
  DE: Ich habe zu tun.



In [2]:
# ---- Write the corpus file for SentencePiece (EN + DE mixed) --------------
N = 40000                                     # subset for fast training
subset = pairs[:N]
en_sents = [en for en, de in subset]
de_sents = [de for en, de in subset]

CORPUS = os.path.join(DATA_DIR, "corpus_deen.txt")
with open(CORPUS, "w", encoding="utf-8") as f:
    for en, de in subset:
        f.write(en + "\n")
        f.write(de + "\n")
print(f"Corpus written: {2*N:,} lines ({N:,} EN + {N:,} DE)")

Corpus written: 80,000 lines (40,000 EN + 40,000 DE)


### Task 1 — Train a shared BPE vocabulary

The training call is given. Train a **shared** BPE model (vocabulary 8000) on the mixed
DE+EN corpus and look at the splits.

**Your task (`# TODO`):** use the loaded processor `sp` to **encode** the two example
sentences (`sp.encode(text, out_type=str)`), and print the token lists. Watch for `▁` —
that is SentencePiece's marker for a space (start of a word).


In [3]:
# ---- Train the BPE model (given) + encode (YOU) ---------------------------
import sentencepiece as spm

MODEL_PREFIX = os.path.join(DATA_DIR, "shared_bpe")
if not os.path.exists(MODEL_PREFIX + ".model"):
    spm.SentencePieceTrainer.train(
        input=CORPUS, model_prefix=MODEL_PREFIX,
        vocab_size=8000, model_type="bpe",
        character_coverage=1.0, input_sentence_size=200000,
        shuffle_input_sentence=True)
sp = spm.SentencePieceProcessor(model_file=MODEL_PREFIX + ".model")
print("Vocabulary size:", sp.get_piece_size())

ex_en = "I don't understand this complicated sentence."
ex_de = "Ich verstehe diesen komplizierten Satz nicht."
print("EN:", sp.encode(ex_en, out_type=str))
print("DE:", sp.encode(ex_de, out_type=str))

Vocabulary size: 8000
EN: ['▁I', '▁don', "'", 't', '▁understand', '▁this', '▁com', 'pl', 'icated', '▁sen', 'ten', 'ce', '.']
DE: ['▁Ich', '▁verstehe', '▁diesen', '▁kompl', 'iz', 'ierten', '▁S', 'atz', '▁nicht', '.']


### Task 2 — Measure fertility

**Fertility** = the average number of subword tokens per (space-separated) word. It measures
how strongly a vocabulary "fragments" a language: 1.0 = every word stays a single token,
higher = more splitting.

**Your task (`# TODO`):** implement `fertility(sentences)`: the sum of subword tokens over
all sentences divided by the sum of words (`text.split()`). Compute it separately for English
and German and compare.


In [4]:
# ---- TASK 2: fertility ---------------------------------------------------
def fertility(sentences):
    n_pieces = sum(len(sp.encode(s, out_type=str)) for s in sentences)
    n_words = sum(len(s.split()) for s in sentences)
    return n_pieces / n_words

# on an evaluation subset (not the first ones -> a little variance)
eval_en = [en for en, de in pairs[100000:105000]]
eval_de = [de for en, de in pairs[100000:105000]]
f_en, f_de = fertility(eval_en), fertility(eval_de)
print(f"Fertility EN (shared vocabulary): {f_en:.3f}")
print(f"Fertility DE (shared vocabulary): {f_de:.3f}")

Fertility EN (shared vocabulary): 1.500
Fertility DE (shared vocabulary): 1.433


### Task 3 — Vocabulary sharing & vocabulary bias

**(a) Sharing:** in a *shared* vocabulary both languages share tokens (digits, punctuation,
common stems/cognates). Measure the share of vocabulary pieces that actually occur in
**both** English **and** German sentences.

**(b) Bias experiment (`# TODO`):** train an **English-dominated** vocabulary (only on EN
sentences) and measure the German fertility with it. Expectation: German gets **fragmented
more** (higher fertility) — exactly the disadvantage that low-resource languages experience
in English-centric LLMs.


In [5]:
# ---- (a) Vocabulary sharing (given) --------------------------------------
def used_pieces(sentences):
    used = set()
    for s in sentences:
        used.update(sp.encode(s, out_type=str))
    return used

en_pieces = used_pieces([en for en, de in subset[:5000]])
de_pieces = used_pieces([de for en, de in subset[:5000]])
shared = en_pieces & de_pieces
print(f"Used pieces EN: {len(en_pieces)},  DE: {len(de_pieces)}")
print(f"Shared (in both): {len(shared)}  "
      f"= {len(shared)/len(en_pieces | de_pieces):.1%} of the used pieces")
print("Examples of shared pieces:", sorted(list(shared))[:20])

Used pieces EN: 1583,  DE: 2532
Shared (in both): 301  = 7.9% of the used pieces
Examples of shared pieces: ['!', '%', "'", ',', '.', '0', '1', '9', ':45', '?', 'N', 'a', 'ab', 'ack', 'ag', 'al', 'am', 'and', 'ant', 'ap']


In [6]:
# ---- (b) TASK 3: vocabulary bias (EN-only vocabulary) --------------------
EN_CORPUS = os.path.join(DATA_DIR, "corpus_en.txt")
with open(EN_CORPUS, "w", encoding="utf-8") as f:
    for en in en_sents:
        f.write(en + "\n")

EN_PREFIX = os.path.join(DATA_DIR, "en_only_bpe")
if not os.path.exists(EN_PREFIX + ".model"):
    spm.SentencePieceTrainer.train(
        input=EN_CORPUS, model_prefix=EN_PREFIX,
        vocab_size=8000, model_type="bpe",
        character_coverage=1.0, input_sentence_size=200000,
        shuffle_input_sentence=True)
sp_en = spm.SentencePieceProcessor(model_file=EN_PREFIX + ".model")

# small helper analogous to task 2, but with an arbitrary model
def fertility_with(model, sentences):
    n_pieces = sum(len(model.encode(s, out_type=str)) for s in sentences)
    n_words = sum(len(s.split()) for s in sentences)
    return n_pieces / n_words

fe_en = fertility_with(sp_en, eval_en)
fe_de = fertility_with(sp_en, eval_de)
print(f"EN-only vocabulary  -> Fertility EN: {fe_en:.3f},  DE: {fe_de:.3f}")
print(f"Shared vocabulary   -> Fertility EN: {f_en:.3f},  DE: {f_de:.3f}")
print(f"\nGerman is split {fe_de/f_de:.2f}x more by the EN-only vocabulary.")

EN-only vocabulary  -> Fertility EN: 1.430,  DE: 2.869
Shared vocabulary   -> Fertility EN: 1.500,  DE: 1.433

German is split 2.00x more by the EN-only vocabulary.


### For comparison — why not words? *(given)*

A word-based vocabulary would have to store every word and fails on OOV. The cell shows the
vocabulary size for pure word tokenization and the **OOV rate** on unseen sentences — the
argument for subwords from script 2.1.


In [7]:
# ---- Word tokenization: vocabulary size & OOV (given) --------------------
from collections import Counter
train_words = Counter(w for en, de in subset for w in (en + " " + de).lower().split())
word_vocab = set(train_words)
print(f"Word vocabulary (DE+EN, {N:,} pairs): {len(word_vocab):,} types")

test_tokens = [w for en, de in pairs[200000:205000] for w in (en + " " + de).lower().split()]
oov = sum(1 for w in test_tokens if w not in word_vocab)
print(f"OOV rate on unseen sentences: {oov/len(test_tokens):.1%}")
print(f"For comparison: the BPE model NEVER has OOV (vocabulary = {sp.get_piece_size()}, "
      f"falls back to characters/bytes if needed).")

Word vocabulary (DE+EN, 40,000 pairs): 20,591 types


OOV rate on unseen sentences: 11.3%
For comparison: the BPE model NEVER has OOV (vocabulary = 8000, falls back to characters/bytes if needed).


## Reflection — reference answers

1. **Subword vs. word.** The word vocabulary is ~20k types (from 40k pairs), and the OOV rate
   on unseen sentences is ~11 % — every inflection, compound and proper name not seen in
   training becomes `<unk>`. The BPE model with 8000 pieces has *no* OOV because its
   vocabulary is closed *below* the word level: any word can be assembled from known subword
   pieces, and in the worst case it falls back to single characters/bytes, which are always
   in the vocabulary. So it trades a small, fixed vocabulary for slightly longer sequences
   but total coverage.

2. **Fertility.** On short everyday Tatoeba sentences EN (≈ 1.5) and DE (≈ 1.4) are close,
   because these sentences use frequent words that the shared 8000-piece vocabulary keeps as
   whole tokens. German's advantage in the compositional dimension only appears on *longer*,
   compound-rich text: a word like "Donaudampfschifffahrtsgesellschaft" is a single
   whitespace word but has no chance of being one BPE token — it is split into many pieces,
   which drives the DE fertility up. Fertility is a property of the *vocabulary–language–text*
   combination, not of the language alone.

3. **Vocabulary bias.** The EN-only vocabulary roughly **doubles** the German fertility (from
   ≈ 1.4 to ≈ 2.9, factor ~2.0), while EN stays low. This is a cost problem because the model
   pays per token: twice as many tokens means twice the sequence length, hence more
   compute/money for the same text, and the fixed context window holds only half as much
   German. It is a fairness problem because the disadvantage falls systematically on the
   language that was underrepresented in the vocabulary — typically the lower-resource one.
   Balanced or temperature-sampled multilingual vocabularies exist precisely to mitigate this.

4. **Sharing.** The shared pieces are typically digits, punctuation, whitespace-initial
   fragments, short function pieces and cross-lingual cognates/proper names (e.g. names,
   internationalisms). Such sharing is the basis for cross-lingual transfer because it gives
   the two languages a **common substrate** in the embedding table: a piece that appears in
   both languages gets *one* embedding trained on both, so a signal learned on the
   high-resource side is already partially available on the low-resource side. Projects 02
   (aligning separate spaces) and 03 (a shared encoder-decoder) build on exactly this idea of
   a shared representation.

> **Reference** (shared vocabulary, short Tatoeba sentences): fertility EN ≈ 1.5, DE ≈ 1.4;
> word vocabulary ~20k types with ~11 % OOV; the EN-only vocabulary **doubles** the German
> fertility (≈ 2.9, factor ~2.0). Exact numbers vary slightly with the subset.
